# Chapter 2.2: Factorization Machines

## Learning Objectives

By the end of this notebook, you will be able to:

1. Understand the **FM formulation** and why it captures pairwise feature interactions
2. Derive the **O(nk) computation trick** that makes FM efficient
3. Implement **Factorization Machines from scratch** in PyTorch
4. Explain **Field-aware FM (FFM)** and its field-specific embedding design
5. Connect FM to matrix factorization as a special case
6. Train and evaluate FM on synthetic CTR data
7. Compare FM against LR baseline on feature interaction modeling

## Prerequisites

- Chapter 2.1 (Logistic Regression & FTRL)
- Basic PyTorch knowledge (tensors, autograd, nn.Module)
- Understanding of embedding layers for categorical features

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USERNAME/rec_system/blob/main/notebooks/part2/chapter_2.2_factorization_machines.ipynb)
[![Download](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/USERNAME/rec_system/main/notebooks/part2/chapter_2.2_factorization_machines.ipynb)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, log_loss

np.random.seed(42)
torch.manual_seed(42)

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

device = torch.device('cpu')
print(f"Using device: {device}")

## 1. The Problem with Manual Feature Crossing

In Chapter 2.1, we saw that logistic regression needs **manual cross features** to capture interactions. But this has problems:

- Requires domain expertise to choose which features to cross
- Quadratic explosion: \(n\) features produce \(O(n^2)\) crosses
- Rare feature combinations have insufficient data to learn reliable weights

**Factorization Machines** (Rendle, 2010) solve this by learning **latent vectors** for each feature and modeling interactions as inner products.

## 2. FM Formulation

A second-order Factorization Machine models the prediction as:

\[ \hat{y}(\mathbf{x}) = w_0 + \sum_{i=1}^{n} w_i x_i + \sum_{i=1}^{n} \sum_{j=i+1}^{n} \langle \mathbf{v}_i, \mathbf{v}_j \rangle x_i x_j \]

where:
- \(w_0\) is the global bias
- \(w_i\) is the weight for feature \(i\) (first-order)
- \(\mathbf{v}_i \in \mathbb{R}^k\) is the latent vector for feature \(i\)
- \(\langle \mathbf{v}_i, \mathbf{v}_j \rangle = \sum_{f=1}^{k} v_{i,f} \cdot v_{j,f}\) is the inner product

> **Concept:** Instead of learning a separate weight \(w_{ij}\) for each pair (which requires observing each pair), FM learns a latent vector \(\mathbf{v}_i\) per feature. The interaction between features \(i\) and \(j\) is \(\langle \mathbf{v}_i, \mathbf{v}_j \rangle\). This allows **generalization** -- even if features \(i\) and \(j\) never co-occur, their interaction can be estimated from their individual interactions with other features.

## 3. The O(nk) Computation Trick

Naively computing all pairwise interactions is \(O(n^2 k)\). The key insight is to reformulate:

\[ \sum_{i=1}^{n} \sum_{j=i+1}^{n} \langle \mathbf{v}_i, \mathbf{v}_j \rangle x_i x_j = \frac{1}{2} \sum_{f=1}^{k} \left[ \left( \sum_{i=1}^{n} v_{i,f} x_i \right)^2 - \sum_{i=1}^{n} v_{i,f}^2 x_i^2 \right] \]

**Proof sketch:**

\[ \left( \sum_i v_{i,f} x_i \right)^2 = \sum_i \sum_j v_{i,f} v_{j,f} x_i x_j = \sum_i v_{i,f}^2 x_i^2 + 2 \sum_{i<j} v_{i,f} v_{j,f} x_i x_j \]

Rearranging:

\[ \sum_{i<j} v_{i,f} v_{j,f} x_i x_j = \frac{1}{2} \left[ \left(\sum_i v_{i,f} x_i\right)^2 - \sum_i v_{i,f}^2 x_i^2 \right] \]

Summing over all \(f\) gives us the \(O(nk)\) formula.

> **Pro Tip:** For sparse inputs (most \(x_i = 0\)), the actual computation is \(O(\bar{n}k)\) where \(\bar{n}\) is the average number of non-zero features, which is very small in CTR prediction.

## 4. Generating Synthetic CTR Data

In [ ]:
def generate_fm_data(n_samples=30000, n_fields=6, seed=42):
    """
    Generate synthetic CTR data with feature interactions.
    Each field is a categorical feature encoded as integer indices.
    """
    rng = np.random.RandomState(seed)
    
    # Field cardinalities (number of unique values per field)
    field_dims = [10, 5, 15, 20, 8, 24]  # user, gender, ad_cat, advertiser, device, hour
    total_features = sum(field_dims)
    
    # Generate categorical indices per field
    data = np.zeros((n_samples, n_fields), dtype=np.int64)
    for f in range(n_fields):
        data[:, f] = rng.randint(0, field_dims[f], n_samples)
    
    # Convert to global indices (offset by cumulative field dims)
    offsets = np.array([0] + list(np.cumsum(field_dims[:-1])))
    global_data = data + offsets[np.newaxis, :]
    
    # Generate labels with interactions
    # Create "true" latent vectors to generate interactions
    true_k = 4
    true_v = rng.randn(total_features, true_k) * 0.3
    true_w = rng.randn(total_features) * 0.2
    
    logits = np.zeros(n_samples)
    for i in range(n_samples):
        feats = global_data[i]
        # Linear part
        logits[i] += np.sum(true_w[feats])
        # Interaction part
        for a in range(len(feats)):
            for b in range(a+1, len(feats)):
                logits[i] += np.dot(true_v[feats[a]], true_v[feats[b]])
    
    logits += -2.0  # shift for ~10% CTR
    probs = 1.0 / (1.0 + np.exp(-logits))
    labels = rng.binomial(1, probs)
    
    print(f"Generated {n_samples} samples with {n_fields} fields, {total_features} total features")
    print(f"Field dims: {field_dims}, CTR: {labels.mean():.4f}")
    
    return global_data, labels, field_dims, offsets

data, labels, field_dims, offsets = generate_fm_data()
total_features = sum(field_dims)

# Split
split = 24000
X_train, X_test = data[:split], data[split:]
y_train, y_test = labels[:split], labels[split:]

## 5. FM Implementation in PyTorch

In [ ]:
class FactorizationMachine(nn.Module):
    """Factorization Machine (Rendle, 2010) implemented in PyTorch."""
    
    def __init__(self, num_features, embedding_dim=8):
        super().__init__()
        self.num_features = num_features
        self.embedding_dim = embedding_dim
        
        # Global bias
        self.bias = nn.Parameter(torch.zeros(1))
        
        # First-order weights (linear part)
        self.linear = nn.Embedding(num_features, 1)
        nn.init.xavier_uniform_(self.linear.weight)
        
        # Latent vectors for interactions
        self.embedding = nn.Embedding(num_features, embedding_dim)
        nn.init.xavier_uniform_(self.embedding.weight)
    
    def forward(self, x):
        """
        x: (batch_size, num_fields) - integer feature indices
        """
        # Bias
        out = self.bias.expand(x.size(0))
        
        # Linear part: sum of w_i for active features
        linear_out = self.linear(x).squeeze(-1)  # (batch, fields)
        out = out + linear_out.sum(dim=1)
        
        # Interaction part using O(nk) trick
        # v_i for each active feature: (batch, fields, k)
        embed = self.embedding(x)  # (batch, fields, k)
        
        # For one-hot features, x_i = 1, so:
        # sum_of_square = (sum_i v_i)^2
        # square_of_sum = sum_i v_i^2
        sum_embed = embed.sum(dim=1)           # (batch, k)
        sum_of_square = sum_embed ** 2          # (batch, k)
        square_of_sum = (embed ** 2).sum(dim=1) # (batch, k)
        
        interaction = 0.5 * (sum_of_square - square_of_sum).sum(dim=1)  # (batch,)
        out = out + interaction
        
        return out  # logits
    
    def predict_proba(self, x):
        """Return probabilities."""
        logits = self.forward(x)
        return torch.sigmoid(logits)

print("FM model defined. Parameters:")
model = FactorizationMachine(total_features, embedding_dim=8)
total_params = sum(p.numel() for p in model.parameters())
print(f"  Total parameters: {total_params}")
print(f"  Linear weights: {total_features}")
print(f"  Embedding parameters: {total_features * 8}")

In [ ]:
def train_model(model, X_train, y_train, X_test, y_test, 
                epochs=20, batch_size=256, lr=0.01, weight_decay=1e-5):
    """Train a PyTorch CTR model and return metrics."""
    
    train_dataset = TensorDataset(
        torch.LongTensor(X_train), 
        torch.FloatTensor(y_train)
    )
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.BCEWithLogitsLoss()
    
    history = {'train_loss': [], 'test_loss': [], 'test_auc': []}
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        n_batches = 0
        
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            n_batches += 1
        
        avg_train_loss = total_loss / n_batches
        
        # Evaluate
        model.eval()
        with torch.no_grad():
            test_x = torch.LongTensor(X_test)
            test_logits = model(test_x)
            test_probs = torch.sigmoid(test_logits).numpy()
            test_loss_val = log_loss(y_test, test_probs)
            test_auc = roc_auc_score(y_test, test_probs)
        
        history['train_loss'].append(avg_train_loss)
        history['test_loss'].append(test_loss_val)
        history['test_auc'].append(test_auc)
        
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1}: train_loss={avg_train_loss:.4f}, "
                  f"test_loss={test_loss_val:.4f}, AUC={test_auc:.4f}")
    
    return history

# Train FM
fm_model = FactorizationMachine(total_features, embedding_dim=8)
fm_history = train_model(fm_model, X_train, y_train, X_test, y_test, epochs=20)

## 6. LR Baseline (No Interactions)

In [ ]:
class LinearModel(nn.Module):
    """Logistic Regression baseline (no feature interactions)."""
    
    def __init__(self, num_features):
        super().__init__()
        self.bias = nn.Parameter(torch.zeros(1))
        self.linear = nn.Embedding(num_features, 1)
        nn.init.xavier_uniform_(self.linear.weight)
    
    def forward(self, x):
        out = self.bias.expand(x.size(0))
        out = out + self.linear(x).squeeze(-1).sum(dim=1)
        return out

lr_model = LinearModel(total_features)
lr_history = train_model(lr_model, X_train, y_train, X_test, y_test, epochs=20)

In [ ]:
# Compare FM vs LR
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, 21)

axes[0].plot(epochs, fm_history['test_loss'], 'r-o', label='FM', linewidth=2, markersize=4)
axes[0].plot(epochs, lr_history['test_loss'], 'b-s', label='LR', linewidth=2, markersize=4)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Test Log Loss')
axes[0].set_title('FM vs LR: Test Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, fm_history['test_auc'], 'r-o', label='FM', linewidth=2, markersize=4)
axes[1].plot(epochs, lr_history['test_auc'], 'b-s', label='LR', linewidth=2, markersize=4)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Test AUC')
axes[1].set_title('FM vs LR: Test AUC')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFinal FM AUC: {fm_history['test_auc'][-1]:.4f}")
print(f"Final LR AUC: {lr_history['test_auc'][-1]:.4f}")
print(f"FM improvement: {fm_history['test_auc'][-1] - lr_history['test_auc'][-1]:.4f}")

## 7. Visualizing Learned Interactions

One advantage of FM is that we can inspect the learned latent vectors to understand feature interactions.

In [ ]:
# Visualize the interaction matrix for features in field 0 vs field 2
with torch.no_grad():
    embeddings = fm_model.embedding.weight.numpy()

# Compute interaction strengths between field 0 (dim=10) and field 2 (dim=15)
field0_start = 0
field0_end = field_dims[0]
field2_start = int(offsets[2])
field2_end = field2_start + field_dims[2]

v_field0 = embeddings[field0_start:field0_end]  # (10, k)
v_field2 = embeddings[field2_start:field2_end]  # (15, k)

interaction_matrix = v_field0 @ v_field2.T  # (10, 15)

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(interaction_matrix, cmap='RdBu_r', aspect='auto')
ax.set_xlabel('Field 2 (Ad Category) Feature Index')
ax.set_ylabel('Field 0 (User) Feature Index')
ax.set_title('Learned Feature Interaction Strengths (FM)')
plt.colorbar(im, ax=ax, label='Interaction Strength')
plt.tight_layout()
plt.show()

## 8. Field-aware Factorization Machine (FFM)

FFM (Juan et al., 2016) extends FM by learning **field-specific** latent vectors. Instead of one vector \(\mathbf{v}_i\) per feature, FFM learns \(\mathbf{v}_{i,f_j}\) -- a different vector for each field \(f_j\) that feature \(i\) interacts with:

\[ \hat{y} = w_0 + \sum_i w_i x_i + \sum_{i=1}^{n} \sum_{j=i+1}^{n} \langle \mathbf{v}_{i, f_j}, \mathbf{v}_{j, f_i} \rangle x_i x_j \]

> **Common Pitfall:** FFM has \(O(n \cdot F \cdot k)\) parameters where \(F\) is the number of fields. This can be much larger than FM. The computation is also \(O(n^2 k)\) since the factorization trick no longer applies.

In [ ]:
class FieldAwareFactorizationMachine(nn.Module):
    """Field-aware FM (Juan et al., 2016)."""
    
    def __init__(self, field_dims, embedding_dim=4):
        super().__init__()
        self.num_fields = len(field_dims)
        self.num_features = sum(field_dims)
        self.embedding_dim = embedding_dim
        
        self.bias = nn.Parameter(torch.zeros(1))
        self.linear = nn.Embedding(self.num_features, 1)
        nn.init.xavier_uniform_(self.linear.weight)
        
        # Field-aware embeddings: one embedding per (feature, target_field) pair
        self.field_embeddings = nn.ModuleList([
            nn.Embedding(self.num_features, embedding_dim)
            for _ in range(self.num_fields)
        ])
        for emb in self.field_embeddings:
            nn.init.xavier_uniform_(emb.weight)
    
    def forward(self, x):
        """
        x: (batch_size, num_fields) - integer feature indices
        """
        batch_size = x.size(0)
        
        # Bias + linear
        out = self.bias.expand(batch_size)
        out = out + self.linear(x).squeeze(-1).sum(dim=1)
        
        # Field-aware interactions
        interaction = torch.zeros(batch_size, device=x.device)
        for i in range(self.num_fields):
            for j in range(i + 1, self.num_fields):
                # Feature i uses embedding for field j, and vice versa
                v_i_fj = self.field_embeddings[j](x[:, i])  # (batch, k)
                v_j_fi = self.field_embeddings[i](x[:, j])  # (batch, k)
                interaction += (v_i_fj * v_j_fi).sum(dim=1)
        
        return out + interaction

# Train FFM
ffm_model = FieldAwareFactorizationMachine(field_dims, embedding_dim=4)
ffm_params = sum(p.numel() for p in ffm_model.parameters())
print(f"FFM parameters: {ffm_params} (vs FM: {total_params})")

ffm_history = train_model(ffm_model, X_train, y_train, X_test, y_test, epochs=20)

## 9. Connection to Matrix Factorization

FM is a **generalization** of matrix factorization (MF). Consider a user-item rating prediction:

- Input: \(\mathbf{x} = [\text{one-hot user}, \text{one-hot item}]\)
- Only two non-zero entries: \(x_u = 1\) and \(x_i = 1\)

The FM interaction term becomes:

\[ \langle \mathbf{v}_u, \mathbf{v}_i \rangle \cdot 1 \cdot 1 = \langle \mathbf{v}_u, \mathbf{v}_i \rangle \]

This is exactly matrix factorization! But FM can also handle:
- Additional side features (user age, item category)
- Multiple feature interactions simultaneously
- Cold-start scenarios (through side information)

> **Concept:** FM subsumes MF, SVD++, and many other factorization models. It unifies them under a single framework by treating all features uniformly.

In [ ]:
# Compare all three models
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(range(1, 21), lr_history['test_auc'], 'b-s', label='LR (no interactions)', 
        linewidth=2, markersize=5)
ax.plot(range(1, 21), fm_history['test_auc'], 'r-o', label='FM (k=8)', 
        linewidth=2, markersize=5)
ax.plot(range(1, 21), ffm_history['test_auc'], 'g-^', label='FFM (k=4)', 
        linewidth=2, markersize=5)

ax.set_xlabel('Epoch')
ax.set_ylabel('Test AUC')
ax.set_title('CTR Prediction: LR vs FM vs FFM')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nFinal AUC - LR: {lr_history['test_auc'][-1]:.4f}, "
      f"FM: {fm_history['test_auc'][-1]:.4f}, "
      f"FFM: {ffm_history['test_auc'][-1]:.4f}")

## 10. Effect of Embedding Dimension

In [ ]:
embed_dims = [2, 4, 8, 16, 32]
dim_results = []

for k in embed_dims:
    torch.manual_seed(42)
    model_k = FactorizationMachine(total_features, embedding_dim=k)
    hist = train_model(model_k, X_train, y_train, X_test, y_test, epochs=15)
    best_auc = max(hist['test_auc'])
    dim_results.append(best_auc)
    print(f"k={k}: best AUC={best_auc:.4f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(embed_dims, dim_results, 'r-o', linewidth=2, markersize=8)
ax.set_xlabel('Embedding Dimension (k)')
ax.set_ylabel('Best Test AUC')
ax.set_title('FM: Effect of Embedding Dimension')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Exercises

### Exercise 1: Implement FM Forward Pass from Scratch

Verify the O(nk) trick by implementing both the naive \(O(n^2k)\) and efficient \(O(nk)\) versions.

In [ ]:
# Exercise 1: Implement both naive and efficient FM interaction computation
# TODO: Implement naive_fm_interaction(V, active_indices) in O(n^2 k)
# TODO: Implement efficient_fm_interaction(V, active_indices) in O(nk)
# TODO: Verify they produce the same result
# TODO: Time both implementations

def naive_fm_interaction(V, indices):
    """Naive O(n^2 k) computation of FM interactions."""
    # YOUR CODE HERE
    result = 0.0
    return result

def efficient_fm_interaction(V, indices):
    """Efficient O(nk) computation using the reformulation trick."""
    # YOUR CODE HERE
    result = 0.0
    return result

# Test
# V = np.random.randn(100, 8)
# indices = [0, 5, 23, 47, 89]
# print(f"Naive: {naive_fm_interaction(V, indices):.6f}")
# print(f"Efficient: {efficient_fm_interaction(V, indices):.6f}")

### Exercise 2: FM with Higher-Order Interactions

In [ ]:
# Exercise 2: Implement a higher-order FM
# The standard FM models 2nd-order interactions. Can we add 3rd-order?
# TODO: Extend the FM class to include an optional 3rd-order interaction term
# Hint: Use the ANOVA kernel decomposition or simply add a small MLP on top
# TODO: Compare 2nd-order FM vs FM+MLP on the synthetic data

# YOUR CODE HERE

### Exercise 3: FM for Cold-Start Recommendation

In [ ]:
# Exercise 3: Demonstrate FM's cold-start advantage over MF
# TODO: Generate user-item data with side features (user age, item category)
# TODO: Train FM with all features, and MF (FM with only user+item) 
# TODO: Remove some items from training, evaluate on those cold-start items
# TODO: Show that FM with side features handles cold-start better

# YOUR CODE HERE

## Summary

In this chapter, we covered:

1. **Factorization Machines** (Rendle, 2010): automatic pairwise feature interactions via latent vectors
2. The **O(nk) computation trick** that makes FM practical for large-scale data
3. **Field-aware FM** (FFM): learning separate embeddings per target field for richer interactions
4. FM as a **generalization of matrix factorization** and many other models

### Key Takeaways

- FM automatically learns feature interactions without manual feature engineering
- The interaction strength between two features is the inner product of their latent vectors
- FM generalizes gracefully to unseen feature combinations
- FFM is more expressive but has more parameters and no O(nk) trick

### What's Next

In Chapter 2.3, we'll combine the **memorization** strength of wide models (like LR with crosses) with the **generalization** ability of deep models in Google's **Wide & Deep Learning** framework.